# torch.compile — Making PyTorch Fast

**What you'll learn in this notebook:**
- What `torch.compile` does under the hood
- Basic usage: compiling functions and models
- The 3-stage pipeline: Dynamo → AOTAutograd → Inductor
- Graph breaks: what they are, how to find and fix them
- Dynamic shapes and recompilation
- Compilation modes and when to use each
- Debugging with `torch._dynamo.explain()`
- Proper benchmarking methodology
- When compile helps and when it doesn't

**Prerequisites:** Basic PyTorch model training, understanding of eager execution.

**All code runs on CPU — no GPU required** (though speedups are most dramatic on GPU).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
print(f"PyTorch version: {torch.__version__}")

## What Does torch.compile Do?

In eager mode, PyTorch executes operations one at a time — like an interpreter. `torch.compile` captures a **graph** of operations and optimizes it as a whole, enabling:

- **Operator fusion**: Combine multiple ops into one kernel (fewer memory reads/writes)
- **Memory planning**: Reuse buffers, reduce allocations
- **Code generation**: Produce optimized Triton/C++ kernels tailored to your specific computation

Think of it as: eager = line-by-line interpreter, compiled = optimizing compiler.

In [ ]:
# Basic usage: compile a function
def my_function(x):
    y = torch.sin(x)
    z = torch.cos(x)
    return y * y + z * z  # Should always be 1.0 (trig identity)

# Eager execution
x = torch.randn(1000)
eager_result = my_function(x)

# Compiled execution
compiled_fn = torch.compile(my_function)
compiled_result = compiled_fn(x)

print(f"Eager result (first 5): {eager_result[:5]}")
print(f"Compiled result (first 5): {compiled_result[:5]}")
print(f"Results match: {torch.allclose(eager_result, compiled_result)}")
print(f"\nNote: First call is slow (compilation). Subsequent calls are fast.")

In [ ]:
# Compile a model (most common usage)
class MLP(nn.Module):
    def __init__(self, dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim),
            nn.LayerNorm(dim),
        )
    
    def forward(self, x):
        return self.net(x) + x  # residual connection

model = MLP(256)
compiled_model = torch.compile(model)

x = torch.randn(32, 256)

# Warmup (triggers compilation)
_ = compiled_model(x)

# Verify correctness
eager_out = model(x)
compiled_out = compiled_model(x)
print(f"Model outputs match: {torch.allclose(eager_out, compiled_out, atol=1e-5)}")
print(f"Output shape: {compiled_out.shape}")

## The 3 Stages: Dynamo → AOTAutograd → Inductor

```
Python code
    ↓
┌─────────────────────────────────────────┐
│ TorchDynamo (graph capture)              │
│ Traces Python bytecode → FX graph        │
│ Handles Python control flow, closures    │
└─────────────────────────────────────────┘
    ↓ FX Graph (forward only)
┌─────────────────────────────────────────┐
│ AOTAutograd (differentiation)            │
│ Generates forward + backward graphs      │
│ Applies decompositions (simplify ops)    │
└─────────────────────────────────────────┘
    ↓ FX Graph (forward + backward)  
┌─────────────────────────────────────────┐
│ Inductor (code generation)               │
│ Generates optimized Triton/C++ kernels   │
│ Operator fusion, memory optimization     │
└─────────────────────────────────────────┘
    ↓
Optimized executable code
```

**Dynamo** handles the "Python problem" — it traces through arbitrary Python code.  
**AOTAutograd** handles the "autograd problem" — it compiles both forward and backward.  
**Inductor** handles the "performance problem" — it generates fast kernels.

## Graph Breaks — The Enemy of Performance

A **graph break** occurs when Dynamo encounters Python code it can't trace through. The compiled graph gets split into multiple smaller graphs with eager execution between them — reducing optimization opportunities.

Common causes: `print()`, data-dependent control flow, unsupported Python constructs.

In [ ]:
# Demo: A function that causes a graph break
import torch._dynamo as dynamo

def function_with_break(x):
    y = x * 2
    print("This causes a graph break!")  # print() breaks the graph
    z = y + 1
    return z

# Use explain() to find graph breaks
explanation = dynamo.explain(function_with_break)(torch.randn(10))
print(f"Number of graph breaks: {explanation.break_reasons}")
print(f"Number of graphs: {len(explanation.graphs)}")

In [ ]:
# Fixed version: remove the print (or use torch._dynamo.config.suppress_errors)
def function_fixed(x):
    y = x * 2
    z = y + 1
    return z

explanation = dynamo.explain(function_fixed)(torch.randn(10))
print(f"Graph breaks after fix: {explanation.break_reasons}")
print(f"Number of graphs: {len(explanation.graphs)}")
print("Clean graph — one continuous compiled region!")

## fullgraph=True — Strict Mode

Use `fullgraph=True` to get an error instead of silent graph breaks. This ensures your entire function compiles as one graph — critical for maximum performance.

In [ ]:
# fullgraph=True catches graph breaks as errors
@torch.compile(fullgraph=True)
def strict_function(x):
    return x * 2 + 1

# This works fine
result = strict_function(torch.randn(5))
print(f"Strict mode works: {result}")

# This would raise an error:
try:
    @torch.compile(fullgraph=True)
    def broken_strict(x):
        y = x * 2
        print("break!")  # Will cause compile error
        return y + 1
    
    broken_strict(torch.randn(5))
except Exception as e:
    print(f"\nfullgraph=True caught the break: {type(e).__name__}")
    print(f"Message (truncated): {str(e)[:200]}...")

## Dynamic Shapes and Recompilation

By default, `torch.compile` assumes fixed tensor shapes. If you pass a different shape, it **recompiles** — which is expensive! Use `dynamic=True` to handle variable shapes.

In [ ]:
dynamo.reset()  # Clear compilation cache

# Without dynamic=True: recompiles for each new shape
call_count = 0
def counting_backend(gm, example_inputs):
    global call_count
    call_count += 1
    return gm

@torch.compile(backend=counting_backend)
def add_one(x):
    return x + 1

# Different shapes trigger recompilation
for size in [10, 20, 30, 10]:  # Note: 10 is repeated
    add_one(torch.randn(size))

print(f"Compilations without dynamic: {call_count}")
print("(Each new shape = new compilation!)")

# With dynamic=True: single compilation handles all shapes
dynamo.reset()
call_count = 0

@torch.compile(backend=counting_backend, dynamic=True)
def add_one_dynamic(x):
    return x + 1

for size in [10, 20, 30, 40, 50]:
    add_one_dynamic(torch.randn(size))

print(f"\nCompilations with dynamic=True: {call_count}")
print("(One compilation handles all sizes!)")

## Compilation Modes

`torch.compile` offers different modes trading off compile time vs. runtime performance:

| Mode | Compile Time | Runtime Speed | Use Case |
|------|-------------|---------------|----------|
| `"default"` | Medium | Good | General use |
| `"reduce-overhead"` | Medium | Better (less framework overhead) | Small models, inference |
| `"max-autotune"` | Very slow | Best | Production deployment |

In [ ]:
dynamo.reset()

class BenchModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
        )
    
    def forward(self, x):
        return self.layers(x)

model = BenchModel()
x = torch.randn(64, 512)

# Compare modes (compilation time)
for mode in ["default", "reduce-overhead"]:
    dynamo.reset()
    start = time.perf_counter()
    compiled = torch.compile(model, mode=mode)
    _ = compiled(x)  # Trigger compilation
    compile_time = time.perf_counter() - start
    print(f"Mode '{mode}': compile time = {compile_time:.2f}s")

print("""
Usage:
  torch.compile(model)                          # default mode
  torch.compile(model, mode="reduce-overhead")  # less Python overhead  
  torch.compile(model, mode="max-autotune")     # tries many kernel configs
""")

## Debugging with torch._dynamo.explain()

`explain()` is your best friend for understanding what the compiler does with your code. It shows graph breaks, the captured graphs, and reasons for breaks.

In [ ]:
dynamo.reset()

class ModelWithIssues(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(32, 32)
        self.history = []  # Mutable state can cause issues
    
    def forward(self, x):
        x = self.linear(x)
        x = F.relu(x)
        self.history.append(x.shape)  # Graph break: mutating Python list
        return x

model = ModelWithIssues()
x = torch.randn(4, 32)

explanation = dynamo.explain(model)(x)
print(f"Graphs: {len(explanation.graphs)}")
print(f"Break reasons: {len(explanation.break_reasons)}")
for i, reason in enumerate(explanation.break_reasons):
    print(f"  Break {i}: {reason}")

## Benchmarking: Compiled vs Eager

Proper benchmarking requires:
1. **Warmup** — first calls compile and are slow
2. **Multiple iterations** — reduce noise
3. **Synchronization** — on GPU, use `torch.cuda.synchronize()`

In [ ]:
dynamo.reset()

def benchmark(fn, x, warmup=5, iterations=50):
    """Proper benchmarking with warmup."""
    # Warmup (compilation happens here)
    for _ in range(warmup):
        fn(x)
    
    # Timed iterations
    start = time.perf_counter()
    for _ in range(iterations):
        fn(x)
    elapsed = time.perf_counter() - start
    return elapsed / iterations * 1000  # ms per call


class LargerModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(*[
            nn.Sequential(nn.Linear(256, 256), nn.GELU(), nn.LayerNorm(256))
            for _ in range(6)
        ])
    
    def forward(self, x):
        return self.layers(x)

model = LargerModel()
x = torch.randn(128, 256)

# Benchmark eager
eager_ms = benchmark(model, x)

# Benchmark compiled
compiled_model = torch.compile(model)
compiled_ms = benchmark(compiled_model, x)

print(f"Eager:    {eager_ms:.3f} ms/call")
print(f"Compiled: {compiled_ms:.3f} ms/call")
print(f"Speedup:  {eager_ms / compiled_ms:.2f}x")
print(f"\nNote: On CPU the speedup is modest. On GPU, expect 1.5-3x for typical models.")

## When Does Compile Help? When Doesn't It?

**Compile helps most when:**
- Model has many fusible operations (element-wise ops, normalization, activation)
- GPU-bound workloads (memory bandwidth limited)
- Repeated calls with same shapes (amortize compile cost)
- Training loops (forward + backward both optimized)

**Compile helps less when:**
- Very small models (compilation overhead > runtime savings)
- Single-call inference (compilation cost not amortized)
- Heavy I/O bound workloads
- Models with many graph breaks (Python-heavy logic)
- Already using heavily optimized custom kernels

In [ ]:
dynamo.reset()

# Example: compile shines with fusible ops
def lots_of_pointwise(x):
    """Many element-wise ops that can be fused into one kernel."""
    x = x * 2
    x = x + 1
    x = torch.relu(x)
    x = x * x
    x = torch.sigmoid(x)
    x = x - 0.5
    x = torch.tanh(x)
    return x

x = torch.randn(1024, 1024)

eager_ms = benchmark(lots_of_pointwise, x)
compiled_ms = benchmark(torch.compile(lots_of_pointwise), x)
print(f"Pointwise-heavy function:")
print(f"  Eager:    {eager_ms:.3f} ms")
print(f"  Compiled: {compiled_ms:.3f} ms")
print(f"  Speedup:  {eager_ms / compiled_ms:.2f}x")

# Example: compile helps less with single large matmul (already optimized)
def single_matmul(x):
    """One big matmul — already calls optimized BLAS."""
    return x @ x.T

dynamo.reset()
eager_ms = benchmark(single_matmul, x)
compiled_ms = benchmark(torch.compile(single_matmul), x)
print(f"\nSingle matmul:")
print(f"  Eager:    {eager_ms:.3f} ms")
print(f"  Compiled: {compiled_ms:.3f} ms")
print(f"  Speedup:  {eager_ms / compiled_ms:.2f}x (minimal — matmul is already fast)")

## Compiled Training Loop

`torch.compile` works seamlessly with training — it compiles both forward and backward passes.

In [ ]:
dynamo.reset()

model = LargerModel()
compiled_model = torch.compile(model)
optimizer = torch.optim.AdamW(compiled_model.parameters(), lr=1e-3)

# Training step — compile captures forward + loss + backward
x = torch.randn(64, 256)
target = torch.randn(64, 256)

print("Compiled training loop:")
for step in range(5):
    optimizer.zero_grad()
    output = compiled_model(x)
    loss = F.mse_loss(output, target)
    loss.backward()
    optimizer.step()
    print(f"  Step {step}: loss = {loss.item():.4f}")

print("\ntorch.compile works with full training — forward, backward, and optimizer!")

## Try It Yourself: Find and Fix a Graph Break

**Exercise:** The model below has a graph break. Use `torch._dynamo.explain()` to find it, then fix the code so it compiles as a single graph.

In [ ]:
dynamo.reset()

# This model has a graph break. Find it and fix it!
class BrokenModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(32, 64)
        self.linear2 = nn.Linear(64, 32)
    
    def forward(self, x):
        x = self.linear1(x)
        x = F.relu(x)
        
        # BUG: This causes a graph break!
        print(f"Shape after linear1: {x.shape}")
        
        x = self.linear2(x)
        return x

# Step 1: Find the break
model = BrokenModel()
x = torch.randn(4, 32)
explanation = dynamo.explain(model)(x)
print(f"Number of breaks: {len(explanation.break_reasons)}")
print(f"Break reason: {explanation.break_reasons}")

# Step 2: Fix it (remove the print or replace with torch.compiler.is_compiling() guard)
class FixedModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(32, 64)
        self.linear2 = nn.Linear(64, 32)
    
    def forward(self, x):
        x = self.linear1(x)
        x = F.relu(x)
        # Removed the print statement!
        x = self.linear2(x)
        return x

dynamo.reset()
fixed_model = FixedModel()
explanation = dynamo.explain(fixed_model)(x)
print(f"\nAfter fix — breaks: {len(explanation.break_reasons)}")
print("Clean compilation!")

## Key Takeaways

1. **`torch.compile`** captures an operation graph and generates optimized code — 1 line to add, often 1.5-3x speedup on GPU
2. **Three stages**: Dynamo (graph capture) → AOTAutograd (backward graph) → Inductor (kernel codegen)
3. **Graph breaks** split the compiled graph — find them with `torch._dynamo.explain()`, fix by removing Python side effects
4. **`fullgraph=True`** raises errors on graph breaks — use it to ensure clean compilation
5. **`dynamic=True`** prevents recompilation when tensor shapes change (critical for variable-length inputs)
6. **Modes**: `"default"` for development, `"reduce-overhead"` for inference, `"max-autotune"` for production
7. **Benchmark properly**: warmup first, time many iterations, compare apples-to-apples
8. **Best gains**: pointwise-heavy code, model training, repeated calls with same shapes
9. **Minimal gains**: single matmuls, I/O-bound code, one-shot inference
10. **Compile is safe**: outputs are numerically equivalent to eager (by design)